In [23]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, xgboost as xgb
if os.path.basename(os.getcwd()) == 'notes': os.chdir('..')
print('工作目录:', os.getcwd())

工作目录: d:\github\prediction


In [ ]:
# ============================================================
# 数据: NetCDF 站点 → 80/15/5 分割 → 预处理 → 全局直接多步特征 (含未来气象)
# 思路: 每行 = [历史PM特征@T] + [未来气象/时间特征@(T+h)] + [horizon h]
#      → 目标 = 线性 PM@(T+h)。模型借"未来气象预报"判断 T+h 是否累积成峰。
#      (与原模型一致地假设已知未来气象; 不递归, 无误差累积)
# 参考: https://xgboost.readthedocs.io/en/stable/python/python_api.html
# ============================================================
import netCDF4 as nc

WEATHER = ['DEWP','HUMI','PRES','TEMP','Iws','precipitation','Iprec']
WEATHER_M6 = [f'{c}_m6' for c in WEATHER]
PM_LAGS = [1,2,3,6,12,24]; MAX_LAG = 24
TIME_COLS = ['hour_sin','hour_cos','month_sin','month_cos','dow_sin','dow_cos']
CBWD_COLS = ['cbwd_cv','cbwd_SE','cbwd_NW','cbwd_SW','cbwd_NE']
ROLL_COLS = ['pm_r6m','pm_r6s','pm_r24m','pm_r24s']
HIST_COLS = [f'pm_t-{l}' for l in PM_LAGS] + ROLL_COLS + ['pm_d_short','pm_d_long','pm_r6x','pm_r24x']  # 14, @T
FUT_COLS  = WEATHER + WEATHER_M6 + TIME_COLS + CBWD_COLS        # 25, @(T+h) 未来气象预报
FEAT = HIST_COLS + FUT_COLS + ['horizon']                       # 40
N_FEAT = len(FEAT)

def load_station(nc_path, idx=0):
    """从 NetCDF 读取站点, 转为 DataFrame (变量映射: K→°C, Pa→hPa, m→mm)"""
    f = nc.Dataset(nc_path)
    t = f.variables['time'][:]
    dt = pd.Timestamp(f.variables['time'].units.split('since ')[1]) + pd.to_timedelta(t, unit='h')
    pm25 = f.variables['PM2.5'][:, idx]
    temp = f.variables['t2m'][:, idx] - 273.15
    dewp = f.variables['d2m'][:, idx] - 273.15
    pres = f.variables['sp'][:, idx] / 100.0
    tp = f.variables['tp'][:, idx] * 1000.0
    u, v = f.variables['u100'][:, idx], f.variables['v100'][:, idx]
    iws = np.sqrt(u**2 + v**2)
    wdir = np.degrees(np.arctan2(-u, -v)) % 360
    cbwd = np.where(iws < 0.5, 'cv', np.where(wdir < 90, 'NE',
           np.where(wdir < 180, 'SE', np.where(wdir < 270, 'SW', 'NW')))).astype('<U2')
    a, b = 17.625, 243.04
    humi = np.clip(100 * np.exp(a*dewp/(dewp+b+1e-10)) / np.exp(a*temp/(temp+b+1e-10)), 0, 100)
    f.close()
    df = pd.DataFrame({'PM_Jingan': pm25, 'PM_US Post': pm25, 'PM_Xuhui': pm25,
        'DEWP': dewp, 'HUMI': humi, 'PRES': pres, 'TEMP': temp,
        'cbwd': cbwd, 'Iws': iws, 'precipitation': tp, 'Iprec': np.cumsum(tp)}, index=dt)
    df.index.name = 'datetime'
    return df.resample('1h').first()

def preprocess(df):
    """单个子集独立预处理: pm_ave(log) → 气象插值 → 周期编码 → rolling"""
    d = df.copy()
    d['pm_ave'] = d[['PM_Jingan','PM_US Post','PM_Xuhui']].mean(axis=1)
    d['pm_ave'] = d['pm_ave'].interpolate(method='time', limit=168).ffill().bfill()
    d['pm_ave'] = np.log1p(d['pm_ave'])  # log 仅用于"特征"稳定; 目标在 build_dataset 里还原为线性
    d[WEATHER] = d[WEATHER].interpolate(method='time').ffill().bfill()
    h = d.index.hour.values.astype(float)
    d['hour_sin'] = np.sin(2*np.pi*h/24); d['hour_cos'] = np.cos(2*np.pi*h/24)
    m = d.index.month.values.astype(float)
    d['month_sin'] = np.sin(2*np.pi*m/12); d['month_cos'] = np.cos(2*np.pi*m/12)
    dw = d.index.dayofweek.values.astype(float)
    d['dow_sin'] = np.sin(2*np.pi*dw/7); d['dow_cos'] = np.cos(2*np.pi*dw/7)
    d['cbwd'] = d['cbwd'].ffill().bfill()
    d['cbwd'] = pd.Categorical(d['cbwd'], categories=['cv','SE','NW','SW','NE'])
    dummies = pd.get_dummies(d['cbwd'], prefix='cbwd').astype(float)
    d = pd.concat([d.drop('cbwd', axis=1), dummies], axis=1)
    d['pm_r6m']  = d['pm_ave'].rolling(6, min_periods=1).mean()
    d['pm_r6s']  = d['pm_ave'].rolling(6, min_periods=1).std().fillna(0)
    d['pm_r24m'] = d['pm_ave'].rolling(24, min_periods=1).mean()
    d['pm_r24s'] = d['pm_ave'].rolling(24, min_periods=1).std().fillna(0)
    for c in WEATHER: d[f'{c}_m6'] = d[c].rolling(6, min_periods=1).mean()
    return d

def hist_matrix(df):
    """历史 PM 特征矩阵 (行 t 仅用 t-1 及更早, 无泄漏). 返回 (n,14) numpy."""
    pm = df['pm_ave']; c = {}
    for l in PM_LAGS: c[f'pm_t-{l}'] = pm.shift(l)
    for col in ROLL_COLS: c[col] = df[col].shift(1)
    c['pm_d_short'] = pm.shift(1) - pm.shift(3)         # 短期差分
    c['pm_d_long']  = pm.shift(1) - pm.shift(24)        # 长期差分
    c['pm_r6x']  = pm.rolling(6,  min_periods=1).max().shift(1)   # 近6h峰值
    c['pm_r24x'] = pm.rolling(24, min_periods=1).max().shift(1)  # 近24h峰值
    return pd.DataFrame(c, index=df.index).values

def build_dataset(df, horizons):
    """全局直接样本: 行(T,h) = hist@T + fut@(T+h) + h, 目标=线性 PM@(T+h).
    horizons 可子采样以加速训练; 返回 X (m,40), y (m,) 线性."""
    hist = hist_matrix(df); fut = df[FUT_COLS].values; ylin = np.expm1(df['pm_ave'].values)
    n = len(df); Xs, ys = [], []
    for h in horizons:
        i0, i1 = MAX_LAG, n - h
        if i1 <= i0: continue
        cnt = i1 - i0
        Xh = np.empty((cnt, N_FEAT))
        Xh[:, :14] = hist[i0:i1]          # 历史 @T
        Xh[:, 14:39] = fut[i0+h:i1+h]     # 未来 @(T+h)
        Xh[:, 39] = h                     # horizon
        Xs.append(Xh); ys.append(ylin[i0+h:i1+h])
    return np.vstack(Xs), np.concatenate(ys)

# 加载 + 分割 + 预处理 (先分割再插值, 杜绝边界数据泄漏)
df_raw = load_station('data3/dataset_yrd.nc', idx=0)
n = len(df_raw); n1 = int(n*0.80); n2 = int(n*0.95)
df_tr = preprocess(df_raw.iloc[:n1])
df_va = preprocess(df_raw.iloc[n1:n2])
df_te = preprocess(df_raw.iloc[n2:])
df_va_ctx = pd.concat([df_tr.iloc[-MAX_LAG:], df_va])   # 验证集带训练集上下文
print(f'训练 {len(df_tr)} | 验证 {len(df_va)} | 测试 {len(df_te)}')

# 构建全局样本 (训练全 horizon; 验证子采样 horizon 加速早停)
X_tr, y_tr = build_dataset(df_tr, list(range(1, 49)))
X_va, y_va = build_dataset(df_va_ctx, list(range(1, 49, 6)))
print(f'全局样本: 训练 {X_tr.shape} | 验证 {X_va.shape} | 特征 {N_FEAT}')

CLIP_HI = float(np.expm1(df_tr['pm_ave'].max()))   # 预测上限 = 训练最大 (线性)
print(f'预测上限 {CLIP_HI:.0f} | 训练目标 max={y_tr.max():.0f} p99={np.quantile(y_tr,0.99):.0f} 中位={np.median(y_tr):.0f}')

In [ ]:
# ============================================================
# XGBoost 训练: 全局直接多步 (单模型, horizon 为特征) + 线性目标
# - 线性目标: MSE 直接优化线性 RMSE, 峰值误差按量级放大 (=内建高值权重),
#   避免 log 变换压缩峰值导致的"无峰值"问题 (已验证比 log 目标显著降 RMSE)
# 参考: https://xgboost.readthedocs.io/en/latest/parameter.html
# ============================================================
dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=FEAT)
dva = xgb.DMatrix(X_va, label=y_va, feature_names=FEAT)
params = {
    'objective':'reg:squarederror',   # 线性 MSE → 直接对齐线性 RMSE 指标
    'eval_metric':'rmse',
    'tree_method':'hist',
    'max_depth':8,
    'eta':0.05,
    'subsample':0.85,'colsample_bytree':0.85,
    'min_child_weight':3,'gamma':0.0,'lambda':0.5,'alpha':0.1,
    'nthread':4,
}
bst = xgb.train(params, dtr, num_boost_round=600,
                evals=[(dtr,'train'),(dva,'eval')],
                early_stopping_rounds=60, verbose_eval=50)
print(f'\n最佳轮次: {bst.best_iteration} | val RMSE (ug/m3): {bst.best_score:.2f}')

# 特征重要性 (部分版本对特定模型绘图可能触发底层异常, 用 try 守护)
fig, ax = plt.subplots(figsize=(10, 6))
try:
    xgb.plot_importance(bst, max_num_features=N_FEAT, ax=ax)
    plt.title(f'Feature Importance ({N_FEAT} features)'); plt.tight_layout(); plt.show()
except Exception as e:
    plt.close(fig); print(f'[特征重要性绘图跳过] {e}')

In [ ]:
# ============================================================
# 评估: 36 组, 每组用 24h 上下文@T + 未来气象@T+h → 一次预测 48h (线性, 无递归)
# 预测已是线性量纲; 同时报告 48h RMSE 与前 24h RMSE (参考)
# ============================================================
N_GROUPS, N_CTX, N_PRED = 36, 24, 48
STRIDE = (len(df_te) - N_CTX - N_PRED) // (N_GROUPS - 1)
hist_te = hist_matrix(df_te); fut_te = df_te[FUT_COLS].values
pm_te = np.expm1(df_te['pm_ave'].values)   # 真值 (线性)

rmses, rmses24 = [], []
fig, axes = plt.subplots(6, 6, figsize=(36, 36)); axes = axes.flatten()
for g in range(N_GROUPS):
    T = g * STRIDE + N_CTX
    Xrow = np.empty((N_PRED, N_FEAT))
    Xrow[:, :14] = hist_te[T]
    Xrow[:, 14:39] = fut_te[T+1:T+1+N_PRED]
    Xrow[:, 39] = np.arange(1, N_PRED+1)
    preds = np.clip(bst.predict(xgb.DMatrix(Xrow, feature_names=FEAT)), 0, CLIP_HI)
    actual = pm_te[T:T+N_PRED]
    err2 = (preds - actual) ** 2
    rmses.append(np.sqrt(err2.mean()))          # 48h RMSE
    rmses24.append(np.sqrt(err2[:24].mean()))   # 前 24h RMSE (参考)
    ax = axes[g]; h = np.arange(1, N_PRED+1)
    ax.plot(h, actual, lw=0.8, label='actual')
    ax.plot(h, preds,  lw=0.8, label='predicted')
    ax.set_title(f'G{g+1} RMSE={rmses[-1]:.1f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_xlabel('hour'); ax.set_ylabel('PM2.5')
plt.suptitle(f'XGBoost Direct — {N_GROUPS} Groups (RMSE 48h)', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

rmses = np.array(rmses); rmses24 = np.array(rmses24)
print(f'\n=== {N_GROUPS} 组 RMSE 汇总 (48h | 24h) ===')
for g in range(N_GROUPS): print(f'  G{g+1:2d}: {rmses[g]:6.2f} | {rmses24[g]:6.2f}')
print(f'\n平均 RMSE(48h) = {rmses.mean():.2f} | 参考 RMSE(24h) = {rmses24.mean():.2f}')
print(f'  正常组 G1-24: 48h={rmses[:24].mean():.2f} 24h={rmses24[:24].mean():.2f} | 峰值组 G25-36: 48h={rmses[24:].mean():.2f} 24h={rmses24[24:].mean():.2f}')